In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import pandas as pd
import altair as alt

In [20]:
!ls

drive  sample_data  Taquilla_df.csv


In [24]:
df = pd.read_csv("Taquilla_df.csv", sep=";")

In [25]:
df

,Año,Película,Género,Subgénero,Actor principal,Director,Productora/estudio,Clasificación,Recaudación internacional,Recaudación doméstica
0,2026,Pegasus 3,Comedy,Sport,Zhang Chi,Han Han,Tianjin Maoyan Weiying Culture Media,NaN,$639.611.740,$1.374.946
1,2026,Project Hail Mary,Science fiction,Thriller,Ryan Gosling,Phil Lord y Christopher Miller,Amazon MGM Studios,PG-13,$254.000.000,$256.218.578
2,2026,The Super Mario Galaxy Movie,Animation,Fantasy,Chris Pratt,Aaron Horvath,Universal Pictures,PG,$321.389.000,$307.200.780
3,2026,Hoppers,Animation,Comedy,Jon Hamm,Daniel Chong,Pixar Animation Studios,PG,$197.300.000,$157.185.920
4,2026,Wuthering Heights,Drama,Romance,Margot Robbie,Emerald Fennell,Warner Bros. Pictures,R,$156.400.000,$84.001.072
...,...,...,...,...,...,...,...,...,...,...
295,1977,Smokey and the Bandit,Action,Adventure,Burt Reynolds,Hal Needham,Universal Pictures,PG,87,$126.737.428
296,1977,Close Encounters of the Third Kind,Drama,Science Fiction,Richard Dreyfuss,Steven Spielberg,Columbia Pictures,PG,"$2.696,00",$116.395.460
297,1977,Saturday Night Fever,Drama,Music,John Travolta,John Badham,Paramount Pictures,R,NaN,$94.213.184
298,1977,A Bridge Too Far,Belic,Drama,Dirk Bogarde,Richard Attenborough,United Artists,PG,NaN,$50.750.000


In [26]:
df.isna().sum().sort_values(ascending=False)

,0
Subgénero,13
Clasificación,5
Recaudación internacional,4
Recaudación doméstica,3
Actor principal,1
Director,1
Película,0
Año,0
Género,0
Productora/estudio,0


In [29]:
df["Taquilla_Total"] = (
    df["Recaudación doméstica"] +
    df["Recaudación internacional"]
)

In [30]:
grouped = (
    df.groupby(["Año", "Clasificación"])["Taquilla_Total"]
    .sum()
    .reset_index()
)

In [55]:
totales_anuales = (
    grouped.groupby("Año")["Taquilla_Total"]
    .sum()
    .reset_index(name="Total_Año")
)

In [33]:
df["Recaudación doméstica"] = (
    df["Recaudación doméstica"]
    .replace(r"[\$,\.]", "", regex=True)
    .astype(float)
)

df["Recaudación internacional"] = (
    df["Recaudación internacional"]
    .replace(r"[\$,\.]", "", regex=True)
    .astype(float)
)

In [34]:
df["Recaudación doméstica"].head()

,Recaudación doméstica
0,1374946.0
1,256218578.0
2,307200780.0
3,157185920.0
4,84001072.0


In [35]:
df["Taquilla_Total"] = (
    df["Recaudación doméstica"] +
    df["Recaudación internacional"]
)

In [36]:
df["Taquilla_Total"].head()

,Taquilla_Total
0,640986686.0
1,510218578.0
2,628589780.0
3,354485920.0
4,240401072.0


In [37]:
df["Taquilla_Total"].dtype

dtype('float64')

In [38]:
grouped = (
    df.groupby(["Año", "Clasificación"])["Taquilla_Total"]
    .sum()
    .reset_index()
)

In [39]:
totales_anuales = (
    grouped.groupby("Año")["Taquilla_Total"]
    .sum()
    .reset_index(name="Total_Año")
)

In [40]:
grouped = grouped.merge(totales_anuales, on="Año")

In [41]:
grouped["Porcentaje"] = (
    grouped["Taquilla_Total"] /
    grouped["Total_Año"]
) * 100

In [42]:
grouped.head(20)

,Año,Clasificación,Taquilla_Total,Total_Año,Porcentaje
0,1977,PG,2.434026e+08,2.434026e+08,100.000000
1,1977,R,0.000000e+00,2.434026e+08,0.000000
2,1978,PG,1.186423e+09,1.328745e+09,89.288998
3,1978,R,1.423219e+08,1.328745e+09,10.711002
4,1979,G,8.956680e+07,6.692492e+08,13.383177
5,1979,PG,1.920117e+08,6.692492e+08,28.690619
6,1979,R,3.876706e+08,6.692492e+08,57.926205
7,1980,PG,8.791808e+08,1.221475e+09,71.976949
8,1980,R,3.422947e+08,1.221475e+09,28.023051
9,1981,PG,1.180165e+09,1.350759e+09,87.370509


In [43]:
grouped = grouped[
    grouped["Clasificación"].isin(["G", "PG", "PG-13", "R"])
]

In [68]:
chart = alt.Chart(grouped).mark_area(
    opacity=0.8
).encode(

    x=alt.X(
    "Año:Q",
    title="Año",
    axis=alt.Axis(
        tickCount=15,
        format="d"
    )
  ),

    y=alt.Y(
        "Porcentaje:Q",
        stack="normalize",
        title="Participación en la taquilla",
        axis=alt.Axis(
            grid=True,
            gridOpacity=0.25
        )
    ),

    color=alt.Color(
        "Clasificación:N",
        sort=["G", "PG", "PG-13", "R"],
        scale=alt.Scale(
            domain=["G", "PG", "PG-13", "R"],
            range=[
                "#F4D35E",
                "#4EA5D9",
                "#EE964B",
                "#3D405B"
            ]
        )
    ),

    tooltip=[
        alt.Tooltip("Año:Q"),
        alt.Tooltip("Clasificación:N"),
        alt.Tooltip("Porcentaje:Q", format=".2f")
    ]

).properties(
    width=900,
    height=500,
    title="Cómo la gran pantalla abandonó el cine adulto"
)

chart

alt.Chart(...)

In [69]:
chart.save("visualizacion.html")

In [59]:
!pip install vl-convert-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 33.2 MB/s eta 0:00:00


In [71]:
chart.save("visualizacion.png")

In [64]:
import os

os.listdir()

['.config',
 'drive',
 '.ipynb_checkpoints',
 'Taquilla_df.csv',
 'visualizacion.png',
 'visualizacion.html',
 'sample_data']

In [72]:
from google.colab import files

files.download("visualizacion.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>